# New Test Implementation:

## Data Processing

In [23]:
import pandas as pd
import numpy as np
from PyKalmanSmoothingGlucose.smoother.smooth_SMBG_data import smooth_smbg_data
from datetime import datetime

def read_and_process_data(file_path):
    # Read the CSV file into a DataFrame
    df = pd.read_csv(file_path)
    #y = df['CGM'].values.astype(float)
    #print(df.head())
    df['date'] = pd.to_datetime(df['date'])
    #t = df['date'].values
    #t_new = datetime
    #smoother = smooth_smbg_data(t,y)
    # Display the first few rows of the DataFrame
    print("Initial DataFrame:")
    print(df.head())
    df.dropna(inplace=True)
    print(df.head(100))
    # Convert categorical columns to numerical using one-hot encoding
    categorical_cols = df.select_dtypes(include=['object']).columns
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    
    # Normalize numerical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    df[numerical_cols] = (df[numerical_cols] - df[numerical_cols].mean()) / df[numerical_cols].std()
    
    return df

file = "../data/OhioT1DM.csv"

read_and_process_data(file)

Initial DataFrame:
                 date  CGM  carbs  bolus  basal  galvanic_skin_response  \
0 2027-04-07 09:20:00  NaN    NaN    NaN    0.6                     NaN   
1 2027-04-07 09:25:00  NaN    NaN    NaN    0.6                     NaN   
2 2027-04-07 09:30:00  NaN    NaN    NaN    0.6                     NaN   
3 2027-04-07 09:35:00  NaN    NaN    NaN    0.6                     NaN   
4 2027-04-07 09:40:00  NaN    NaN    NaN    0.6                     NaN   

   skin_temp  acceleration  workout_intensity  workout_duration  is_test  \
0        NaN           NaN                NaN               NaN    False   
1        NaN           NaN                NaN               NaN    False   
2        NaN           NaN                NaN               NaN    False   
3        NaN           NaN                NaN               NaN    False   
4        NaN           NaN                NaN               NaN    False   

  insulin_type gender   id  heartrate  air_temp  steps  
0      humalog  

,date,CGM,carbs,bolus,basal,galvanic_skin_response,skin_temp,acceleration,workout_intensity,workout_duration,is_test,id,heartrate,air_temp,steps


In [ ]:
# Step 1: Data Preparation
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# Load the dataset
url = "../data/OhioT1DM.csv"
df = pd.read_csv(url, usecols=[1])
df.dropna(inplace=True)
data = df['CGM'].values.astype('float32')


# Normalize the data to [0,1]
scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(data.reshape(-1, 1))

# Split into Training and Testing Sets
train_size = int(len(dataset) * 0.67)
test_size = len(dataset) - train_size

train, test = dataset[0:train_size], dataset[train_size:]

# Create Dataset Function
def create_dataset(dataset, look_back=1):
    X, Y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:(i + look_back), 0])
        Y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(Y)

look_back = 12
trainX, trainY = create_dataset(train, look_back)
testX, testY = create_dataset(test, look_back)

# Reshape input to be [samples, time steps, features]
trainX = np.reshape(trainX, (trainX.shape[0], look_back, 1))
testX = np.reshape(testX, (testX.shape[0], look_back, 1))

# Step 2: Building the LSTM Model
from keras.models import Sequential
from keras.layers import LSTM, Dense

model = Sequential()
model.add(LSTM(units=128, input_shape=(look_back, 1)))
model.add(Dense(units=64, activation='relu'))
model.add(Dense(units=1))

# Step 3: Compiling the Model
model.compile(loss='mean_squared_error', optimizer='adam')

# Step 4: Training the Model
model.fit(trainX, trainY, epochs=50, batch_size=32, verbose=2)

# Step 5: Making Predictions
trainPredict = model.predict(trainX)
testPredict = model.predict(testX)

# Invert predictions to original scale
trainPredict = scaler.inverse_transform(trainPredict)
trainY_inv = scaler.inverse_transform([trainY])

testPredict = scaler.inverse_transform(testPredict)
testY_inv = scaler.inverse_transform([testY])


ValueError: Expected 2D array, got 1D array instead:
array=[142. 142. 142. ... 182. 180. 177.].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [5]:
# Step 6: Evaluating the Model 
import math
from sklearn.metrics import mean_squared_error

# Calculate root mean squared error
trainScore = math.sqrt(mean_squared_error(trainY_inv[0], trainPredict[:,0]))
print('Train RMSE: %.2f' % (trainScore))

testScore = math.sqrt(mean_squared_error(testY_inv[0], testPredict[:,0]))
print('Test RMSE: %.2f' % (testScore))

ValueError: Input contains NaN.